In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt


def accuracy(model, dataloader):  # 평가 함수 정의
    cnt = 0
    acc = 0

    for data in dataloader:
        inputs, labels = data

        preds = model(inputs)
        preds = torch.argmax(preds, dim=-1)

        cnt += labels.shape[0]
        acc += (labels == preds).sum().item()

    return acc / cnt

## 두개의 acc 시각화 함수
def plot_acc(acc1, acc2, label1, label2):
    x = np.arange(len(acc1))
    plt.plot(x, acc1, label=label1)
    plt.plot(x, acc2, label=label2)
    plt.legend()
    plt.show()

## 두 모델 평가 후 시각화 함수, model2를 지정하지 않으면 model1 사용, 평가함수도 지정 가능
def acc_and_plot(model1, dataloader1, dataloader2, label1="train", label2="test", model2 = None, acc_func = accuracy):
    if model2 is None:
        model2 = model1

    model1_acc_list = []
    model2_acc_list = []

    model1_acc = acc_func(model1, dataloader1)
    model1_acc_list.append(model1_acc)
    model2_acc = acc_func(model2, dataloader2)
    model2_acc_list.append(model2_acc)

    plot_acc(model1_acc_list, model2_acc_list, label1 = label1, label2 = label2)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

class AccuracyMonitor:
    def __init__(self, models, dataloaders, labels, title="Model Accuracies", accuracy_fn=None, device=None):
        """
        models: 모델 리스트
        dataloaders: 데이터로더 리스트
        labels: 모델 및 데이터로더에 대한 레이블 리스트
        title: 그래프 제목
        accuracy_fn: 사용자 정의 정확도 함수 (default_accuracy를 기본값으로 사용)
        device: 모델과 데이터를 이동시킬 장치 (CPU 또는 GPU)
        """
        if not (len(models) == len(dataloaders) == len(labels)):
            raise ValueError("models, dataloaders, labels는 모두 같은 길이를 가져야 합니다.")

        self.models = models
        self.dataloaders = dataloaders
        self.labels = labels
        self.title = title
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.acc_lists = [[] for _ in labels]
        self.accuracy_fn = accuracy_fn if accuracy_fn else self.default_accuracy

    def default_accuracy(self, model, dataloader):
        """
        기본 정확도 계산 함수. 
        """
        cnt = 0
        acc = 0
        model.eval()
        model.to(self.device)
        with torch.no_grad():
            for data in dataloader:
                inputs, labels = data
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                preds = model(inputs)
                preds = torch.argmax(preds, dim=-1)
                cnt += labels.shape[0]
                acc += (labels == preds).sum().item()
        model.train()
        return acc / cnt if cnt > 0 else 0.0

    def update_accuracies(self, verbose=False):
        """
        models와 dataloaders를 평가하고 acc_lists에 기록.
        verbose: True인 경우 정확도 업데이트 로그 출력.
        """
        for i, (model, dataloader) in enumerate(zip(self.models, self.dataloaders)):
            acc = self.accuracy_fn(model, dataloader, **kwargs)
            self.acc_lists[i].append(acc)
            if verbose:
                print(f"Updated Accuracy ({self.labels[i]}): {acc:.4f}")

    def plot(self, epoch=None, save_path=None):
        """
        정확도 그래프를 출력하고 저장하는 기능.
        epoch: 그래프 제목에 표시할 에폭 정보
        save_path: 그래프를 저장할 경로 (None이면 저장하지 않음)
        """
        if not self.acc_lists or len(self.acc_lists[0]) == 0:
            print("No accuracies to plot")
            return

        x = np.arange(len(self.acc_lists[0]))  # 에폭 수만큼 x축 생성

        plt.figure(figsize=(10, 6))  # 그래프 크기 조정
        for acc_list, label in zip(self.acc_lists, self.labels):
            plt.plot(x, acc_list, label=label, marker='o')

        title = f"{self.title} at Epoch {epoch}" if epoch is not None else self.title
        plt.title(title)
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.grid(True)
        plt.legend()
        plt.ylim(0.0, 1.0)

        if save_path:
            plt.savefig(save_path)
            print(f"Plot saved to {save_path}")
        else:
            plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

class AccuracyMonitor:
    def __init__(self, models, dataloaders, labels, title="Model Accuracies", accuracy_fn=None):
        """
        models: 모델 리스트
        dataloaders: 데이터로더 리스트
        labels: 모델 및 데이터로더에 대한 레이블 리스트
        title: 그래프 제목
        accuracy_fn: 사용자 정의 정확도 함수 (default_accuracy를 기본값으로 사용)
        """
        if not (len(models) == len(dataloaders) == len(labels)):
            raise ValueError("models, dataloaders, labels는 모두 같은 길이를 가져야 합니다.")

        self.models = models
        self.dataloaders = dataloaders
        self.labels = labels
        self.title = title
        self.acc_lists = [[] for _ in labels]
        self.accuracy_fn = accuracy_fn if accuracy_fn else self.default_accuracy

    def default_accuracy(self, model, dataloader, **kwargs):
        """
        기본 정확도 계산 함수. **kwargs를 통해 추가 옵션 지원.
        """
        cnt = 0
        acc = 0
        model.eval()
        with torch.no_grad():
            for data in dataloader:
                inputs, labels = data
                preds = model(inputs)
                preds = torch.argmax(preds, dim=-1)
                cnt += labels.shape[0]
                acc += (labels == preds).sum().item()
        model.train()
        return acc / cnt if cnt > 0 else 0.0

    def update_accuracies(self, verbose=False, **kwargs):
        """
        models와 dataloaders를 평가하고 acc_lists에 기록.
        verbose: True인 경우 정확도 업데이트 로그 출력.
        """
        for i, (model, dataloader) in enumerate(zip(self.models, self.dataloaders)):
            acc = self.accuracy_fn(model, dataloader, **kwargs)
            self.acc_lists[i].append(acc)
            if verbose:
                print(f"Updated Accuracy ({self.labels[i]}): {acc:.4f}")

    def plot(self, epoch=None, save_path=None):
        """
        정확도 그래프를 출력하고 저장하는 기능.
        epoch: 그래프 제목에 표시할 에폭 정보
        save_path: 그래프를 저장할 경로 (None이면 저장하지 않음)
        """
        if not self.acc_lists or len(self.acc_lists[0]) == 0:
            print("No accuracies to plot")
            return

        x = np.arange(len(self.acc_lists[0]))  # 에폭 수만큼 x축 생성

        plt.figure(figsize=(10, 6))  # 그래프 크기 조정
        for acc_list, label in zip(self.acc_lists, self.labels):
            plt.plot(x, acc_list, label=label, marker='o')

        title = f"{self.title} at Epoch {epoch}" if epoch is not None else self.title
        plt.title(title)
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.grid(True)
        plt.legend()
        plt.ylim(0.0, 1.0)

        if save_path:
            plt.savefig(save_path)
            print(f"Plot saved to {save_path}")
        else:
            plt.show()
